# VQA_trainA.ipynb
Self-contained Architecture A: A1 LSTM decoder và A2 Transformer decoder.


In [ ]:
#Cell_0 - Install/import
!pip install -q transformers sentencepiece torchvision tqdm matplotlib
import os, json, random, math, sys
from pathlib import Path
from typing import Any, Dict, List, Sequence
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer
import matplotlib.pyplot as plt


In [ ]:
# !ls /kaggle/input/datasets/khangnhut/vqa-vietnameses-food-dataset/vietnamese_food_images_2k5

In [ ]:
#Cell_1 - Config
DATA_DIR = Path('/kaggle/input/datasets/khangnhut/vqa-vietnameses-food-dataset')
OUT_DIR = Path('/kaggle/working/checkpoints_A'); OUT_DIR.mkdir(exist_ok=True)
IMAGE_MODEL = 'google/vit-base-patch16-224-in21k'
TEXT_MODEL = 'vinai/phobert-base'
BATCH_SIZE = 16
EPOCHS = 10 #20
LR = 3e-5 #1e-4 
MAX_Q_LEN = 64
MAX_A_LEN = 12
FEATURE_DIM = 512
SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


In [ ]:
#Cell_2 - Tokenizer câu trả lời
PAD_TOKEN='<pad>'; BOS_TOKEN='<bos>'; EOS_TOKEN='<eos>'; UNK_TOKEN='<unk>'
class SimpleAnswerTokenizer:
    def __init__(self, max_vocab=8000):
        self.stoi={PAD_TOKEN:0,BOS_TOKEN:1,EOS_TOKEN:2,UNK_TOKEN:3}; self.itos=[PAD_TOKEN,BOS_TOKEN,EOS_TOKEN,UNK_TOKEN]; self.max_vocab=max_vocab
    @property
    def pad_id(self): return self.stoi[PAD_TOKEN]
    @property
    def bos_id(self): return self.stoi[BOS_TOKEN]
    @property
    def eos_id(self): return self.stoi[EOS_TOKEN]
    @property
    def vocab_size(self): return len(self.itos)
    def tokenize(self, text): return str(text or '').strip().split()
    def fit(self, answers):
        from collections import Counter
        c=Counter(t for a in answers for t in self.tokenize(a))
        for tok,_ in c.most_common(self.max_vocab-4):
            if tok not in self.stoi: self.stoi[tok]=len(self.itos); self.itos.append(tok)
    def encode(self, text, max_len):
        ids=[self.bos_id]+[self.stoi.get(t,self.stoi[UNK_TOKEN]) for t in self.tokenize(text)]+[self.eos_id]
        ids=ids[:max_len]
        if ids[-1]!=self.eos_id: ids[-1]=self.eos_id
        return ids+[self.pad_id]*(max_len-len(ids))
    def decode(self, ids):
        out=[]
        for i in ids:
            i=int(i)
            if i==self.eos_id: break
            if i in {self.pad_id,self.bos_id}: continue
            out.append(self.itos[i] if i < len(self.itos) else UNK_TOKEN)
        return ' '.join(out)
    def save(self,path): Path(path).write_text(json.dumps({'itos':self.itos},ensure_ascii=False,indent=2),encoding='utf-8')


In [ ]:
#Cell_3 - Dataset/DataLoader cho Encoder input
IMAGE_DIR_CANDIDATES = [
    DATA_DIR / 'vietnamese_food_images_2k5',
    DATA_DIR / 'vietnamese_food_images',
    DATA_DIR / 'vietnamese_food_images' / 'vietnamese_food_images',
    DATA_DIR / 'vietnamese_food_images_2k5' / 'vietnamese_food_images_2k5',
]

def resolve_image_path(raw_path, image_id=None):
    p = Path(str(raw_path))
    candidates = []
    if p.is_absolute():
        candidates.append(p)
    else:
        candidates.extend([DATA_DIR / p, p])
    candidates.extend([d / p.name for d in IMAGE_DIR_CANDIDATES])
    if image_id is not None:
        candidates.extend([d / f'image_{image_id}.jpg' for d in IMAGE_DIR_CANDIDATES])
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(f'Cannot resolve image_path={raw_path}. Tried: {[str(c) for c in candidates[:8]]} ...')

class VQADataset(Dataset):
    def __init__(self, json_path, qtok, atok, iproc, max_q_len=64, max_a_len=12):
        self.rows=json.loads(Path(json_path).read_text(encoding='utf-8')); self.qtok=qtok; self.atok=atok; self.iproc=iproc; self.max_q_len=max_q_len; self.max_a_len=max_a_len
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        r=self.rows[idx]
        image_path = resolve_image_path(r.get('image_path', ''), r.get('image_id'))
        img=Image.open(image_path).convert('RGB')
        pix=self.iproc(images=img, return_tensors='pt')['pixel_values'].squeeze(0)
        q=self.qtok(r['question'], padding='max_length', truncation=True, max_length=self.max_q_len, return_tensors='pt')
        return {'pixel_values':pix, 'input_ids':q['input_ids'].squeeze(0), 'attention_mask':q['attention_mask'].squeeze(0), 'target_ids':torch.tensor(self.atok.encode(r['answer'], self.max_a_len)), 'answer':r['answer'], 'question_type':r.get('question_type','unknown')}
def collate_vqa(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'target_ids': torch.stack([b['target_ids'] for b in batch]),
        'answer': [b['answer'] for b in batch],
        'question_type': [b['question_type'] for b in batch],
    }


In [ ]:
#Cell_4 - Image Encoder, Text Encoder, Fusion
class GatedFusion(nn.Module):
    def __init__(self, dim=512):
        super().__init__(); self.gate=nn.Sequential(nn.Linear(dim*2,dim), nn.Sigmoid()); self.norm=nn.LayerNorm(dim)
    def forward(self, img, txt):
        g=self.gate(torch.cat([img,txt],-1)); return self.norm(g*img+(1-g)*txt)
class ImageTextEncoder(nn.Module):
    def __init__(self, image_model=IMAGE_MODEL, text_model=TEXT_MODEL, feature_dim=512, freeze=True):
        super().__init__(); self.image_encoder=AutoModel.from_pretrained(image_model); self.text_encoder=AutoModel.from_pretrained(text_model)
        if freeze:
            for p in self.image_encoder.parameters(): p.requires_grad=False
            for p in self.text_encoder.parameters(): p.requires_grad=False
        self.img_proj=nn.Linear(self.image_encoder.config.hidden_size, feature_dim); self.txt_proj=nn.Linear(self.text_encoder.config.hidden_size, feature_dim); self.fusion=GatedFusion(feature_dim)
    def forward(self, pixel_values, input_ids, attention_mask):
        io=self.image_encoder(pixel_values=pixel_values); img=(io.pooler_output if getattr(io,'pooler_output',None) is not None else io.last_hidden_state[:,0])
        to=self.text_encoder(input_ids=input_ids, attention_mask=attention_mask); txt=to.last_hidden_state[:,0]
        return self.fusion(self.img_proj(img), self.txt_proj(txt))


In [ ]:
#Cell_5 - LSTMDecoder A1, TransformerDecoder A2, VQAModelA
class LSTMDecoder(nn.Module):
    def __init__(self, vocab, feature_dim=512, emb=256, hidden=512):
        super().__init__(); self.emb=nn.Embedding(vocab,emb,padding_idx=0); self.lstm=nn.LSTM(emb+feature_dim,hidden,batch_first=True); self.fc=nn.Linear(hidden,vocab)
    def forward(self, fused, target_ids):
        x=target_ids[:,:-1]; e=self.emb(x); f=fused.unsqueeze(1).expand(-1,e.size(1),-1); o,_=self.lstm(torch.cat([e,f],-1)); return self.fc(o)
class TransformerDecoder(nn.Module):
    def __init__(self, vocab, feature_dim=512, emb=256, nhead=4, layers=2, max_len=32):
        super().__init__(); self.emb=nn.Embedding(vocab,emb,padding_idx=0); self.pos=nn.Parameter(torch.zeros(1,max_len,emb)); self.mem=nn.Linear(feature_dim,emb); layer=nn.TransformerDecoderLayer(emb,nhead,emb*4,batch_first=True); self.dec=nn.TransformerDecoder(layer,layers); self.fc=nn.Linear(emb,vocab)
    def forward(self, fused, target_ids):
        x=target_ids[:,:-1]; e=self.emb(x)+self.pos[:,:x.size(1),:]; mask=torch.triu(torch.full((x.size(1),x.size(1)),float('-inf'),device=x.device),1); return self.fc(self.dec(e,self.mem(fused).unsqueeze(1),tgt_mask=mask))
class VQAModelA(nn.Module):
    def __init__(self, vocab, decoder='lstm'):
        super().__init__(); self.encoder=ImageTextEncoder(); self.decoder=LSTMDecoder(vocab) if decoder=='lstm' else TransformerDecoder(vocab)
    def forward(self, pixel_values, input_ids, attention_mask, target_ids): return self.decoder(self.encoder(pixel_values,input_ids,attention_mask), target_ids)
    @torch.no_grad()
    def generate(self, pixel_values, input_ids, attention_mask, bos_id, eos_id, max_len=12):
        self.eval(); fused=self.encoder(pixel_values,input_ids,attention_mask); gen=torch.full((pixel_values.size(0),1),bos_id,dtype=torch.long,device=pixel_values.device)
        for _ in range(max_len-1):
            dec_in=torch.cat([gen, torch.full_like(gen[:,:1], eos_id)], 1); nxt=self.decoder(fused,dec_in)[:,-1].argmax(-1,keepdim=True); gen=torch.cat([gen,nxt],1)
            if torch.all(nxt.squeeze(1)==eos_id): break
        return gen


In [ ]:
#Cell_6 - Train/evaluate functions with full per-epoch validation metrics
# Per-epoch: train_loss, val_loss, exact, soft, BLEU, ROUGE-L, METEOR, LLM-judge rule. BERTScore chỉ tính final test vì chậm.
import re, pandas as pd
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score
    _smooth = SmoothingFunction().method1
except Exception as e:
    sentence_bleu = None; meteor_score = None; _smooth = None
    print('[WARN] NLTK metrics unavailable:', e)
try:
    from rouge_score import rouge_scorer
    _rouge_epoch = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=False)
except Exception as e:
    _rouge_epoch = None
    print('[WARN] rouge-score unavailable:', e)

def set_seed(seed): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
def norm_text_epoch(x): return re.sub(r'\s+', ' ', str(x or '').strip().lower())
def exact(preds, refs): return sum(norm_text_epoch(p)==norm_text_epoch(r) for p,r in zip(preds,refs))/max(len(refs),1)
def soft(preds, refs):
    s=0
    for p,r in zip(preds,refs):
        p=norm_text_epoch(p); r=norm_text_epoch(r)
        if p==r: s += 1.0
        elif p and r and (p in r or r in p): s += 0.5
        else:
            ps, rs = set(p.split()), set(r.split())
            s += len(ps & rs) / max(len(ps | rs), 1)
    return s/max(len(refs),1)
def bleu_one_epoch(pred, ref):
    if sentence_bleu is None: return 0.0
    p=norm_text_epoch(pred).split(); r=norm_text_epoch(ref).split()
    return 0.0 if not p or not r else float(sentence_bleu([r], p, smoothing_function=_smooth))
def rouge_l_one_epoch(pred, ref):
    return 0.0 if _rouge_epoch is None else float(_rouge_epoch.score(norm_text_epoch(ref), norm_text_epoch(pred))['rougeL'].fmeasure)
def meteor_one_epoch(pred, ref):
    if meteor_score is None: return 0.0
    p=norm_text_epoch(pred).split(); r=norm_text_epoch(ref).split()
    return 0.0 if not p or not r else float(meteor_score([r], p))
def llm_judge_rule_epoch(pred, ref):
    p, r = norm_text_epoch(pred), norm_text_epoch(ref)
    if p==r: return 1.0
    if p and r and (p in r or r in p): return 0.7
    ps, rs = set(p.split()), set(r.split())
    return 0.5 if len(ps & rs) / max(len(ps | rs), 1) >= 0.5 else 0.0
def summarize_epoch_metrics(preds, refs):
    if not preds: return {k:0.0 for k in ['exact','soft','bleu','rougeL','meteor','llm_judge_rule']}
    rows=[{'exact':float(norm_text_epoch(p)==norm_text_epoch(r)),'soft':soft([p],[r]),'bleu':bleu_one_epoch(p,r),'rougeL':rouge_l_one_epoch(p,r),'meteor':meteor_one_epoch(p,r),'llm_judge_rule':llm_judge_rule_epoch(p,r)} for p,r in zip(preds,refs)]
    return {k:float(np.mean([x[k] for x in rows])) for k in rows[0]}
def save_history_files(hist, out):
    out=Path(out); out.mkdir(parents=True, exist_ok=True)
    (out/'history.json').write_text(json.dumps(hist,ensure_ascii=False,indent=2),encoding='utf-8')
    pd.DataFrame(hist).to_csv(out/'history.csv',index=False)
def run_train(decoder):
    set_seed(SEED); iproc=AutoImageProcessor.from_pretrained(IMAGE_MODEL); qtok=AutoTokenizer.from_pretrained(TEXT_MODEL, use_fast=False)
    answers=[r['answer'] for r in json.loads((DATA_DIR/'train.json').read_text(encoding='utf-8'))]; atok=SimpleAnswerTokenizer(); atok.fit(answers)
    train_ds=VQADataset(DATA_DIR/'train.json',qtok,atok,iproc,MAX_Q_LEN,MAX_A_LEN); val_ds=VQADataset(DATA_DIR/'val.json',qtok,atok,iproc,MAX_Q_LEN,MAX_A_LEN)
    train_dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,collate_fn=collate_vqa); val_dl=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,collate_fn=collate_vqa)
    model=VQAModelA(atok.vocab_size,decoder).to(DEVICE); opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LR,weight_decay=1e-2); ce=nn.CrossEntropyLoss(ignore_index=atok.pad_id); scaler=GradScaler('cuda',enabled=DEVICE.type=='cuda')
    out=OUT_DIR/decoder; out.mkdir(parents=True,exist_ok=True); hist=[]; best=-1
    for ep in range(1,EPOCHS+1):
        model.train(); total=0; n=0
        for b in tqdm(train_dl, desc=f'{decoder} train e{ep}'):
            pix=b['pixel_values'].to(DEVICE); ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE); tgt=b['target_ids'].to(DEVICE); opt.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=DEVICE.type=='cuda'):
                logits=model(pix,ids,mask,tgt); loss=ce(logits.reshape(-1,logits.size(-1)), tgt[:,1:].reshape(-1))
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt); gn=nn.utils.clip_grad_norm_(model.parameters(),0.5)
            if torch.isfinite(gn): scaler.step(opt)
            scaler.update(); total+=float(loss.item())*pix.size(0); n+=pix.size(0)
        model.eval(); preds=[]; refs=[]; vloss=0; vn=0
        with torch.no_grad():
            for b in tqdm(val_dl, desc=f'{decoder} val e{ep}'):
                pix=b['pixel_values'].to(DEVICE); ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE); tgt=b['target_ids'].to(DEVICE)
                logits=model(pix,ids,mask,tgt); loss=ce(logits.reshape(-1,logits.size(-1)), tgt[:,1:].reshape(-1)); gen=model.generate(pix,ids,mask,atok.bos_id,atok.eos_id,MAX_A_LEN)
                preds += [atok.decode(x.tolist()) for x in gen]; refs += b['answer']; vloss += float(loss.item())*pix.size(0); vn += pix.size(0)
        m=summarize_epoch_metrics(preds,refs)
        rec={'epoch':ep,'train_loss':total/max(n,1),'val_loss':vloss/max(vn,1),'val_exact':m['exact'],'val_soft':m['soft'],'val_bleu':m['bleu'],'val_rougeL':m['rougeL'],'val_meteor':m['meteor'],'val_llm_judge_rule':m['llm_judge_rule']}
        print(json.dumps(rec,ensure_ascii=False,indent=2)); hist.append(rec); save_history_files(hist,out)
        if rec['val_soft']>best:
            best=rec['val_soft']; torch.save({'model_state':model.state_dict(),'decoder':decoder,'vocab':atok.itos,'best_epoch':ep,'best_val_soft':best}, out/'best.pt')
    return hist


In [ ]:
#Cell_7 - Train A1 LSTM
# history_A1 = run_train('lstm')


In [ ]:
#Cell_8 - Train A2 Transformer
history_A2 = run_train('transformer')


In [ ]:
#Cell_9 - Per-model history plots after training
# Cell này chỉ vẽ biểu đồ cho model đang train trong notebook hiện tại, dựa trên history riêng đã lưu.
import pandas as pd
import seaborn as sns

CURRENT_DECODER = 'lstm'
CURRENT_MODEL_NAME = 'A1 LSTM'
MODEL_OUT_DIR = OUT_DIR / CURRENT_DECODER
HISTORY_JSON = MODEL_OUT_DIR / 'history.json'
HISTORY_CSV = MODEL_OUT_DIR / 'history.csv'
PLOT_DIR = MODEL_OUT_DIR / 'plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def load_current_history():
    if HISTORY_JSON.exists():
        return pd.DataFrame(json.loads(HISTORY_JSON.read_text(encoding='utf-8')))
    if HISTORY_CSV.exists():
        return pd.read_csv(HISTORY_CSV)
    if 'history_A1' in globals():
        return pd.DataFrame(history_A1)
    raise FileNotFoundError(f'Không tìm thấy history tại {HISTORY_JSON} hoặc {HISTORY_CSV}. Hãy chạy cell train trước.')

hist_df = load_current_history()
hist_df.to_csv(MODEL_OUT_DIR / f'{CURRENT_DECODER}_history_export.csv', index=False)
display(hist_df)

plt.figure(figsize=(9,4))
plt.plot(hist_df['epoch'], hist_df['train_loss'], marker='o', label='train_loss')
plt.plot(hist_df['epoch'], hist_df['val_loss'], marker='o', label='val_loss')
plt.title(f'{CURRENT_MODEL_NAME} - Loss curves')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(PLOT_DIR / f'{CURRENT_DECODER}_loss_curves.png', dpi=180)
plt.show()

plt.figure(figsize=(9,4))
if 'val_exact' in hist_df: plt.plot(hist_df['epoch'], hist_df['val_exact'], marker='o', label='val_exact')
if 'val_soft' in hist_df: plt.plot(hist_df['epoch'], hist_df['val_soft'], marker='o', label='val_soft')
plt.title(f'{CURRENT_MODEL_NAME} - Validation accuracy curves')
plt.xlabel('Epoch'); plt.ylabel('Score'); plt.ylim(0,1); plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
plt.savefig(PLOT_DIR / f'{CURRENT_DECODER}_accuracy_curves.png', dpi=180)
plt.show()

metric_cols = [c for c in ['val_bleu','val_rougeL','val_meteor','val_llm_judge_rule'] if c in hist_df.columns]
if metric_cols:
    plt.figure(figsize=(10,5))
    for c in metric_cols:
        plt.plot(hist_df['epoch'], hist_df[c], marker='o', label=c)
    plt.title(f'{CURRENT_MODEL_NAME} - Validation generation metrics')
    plt.xlabel('Epoch'); plt.ylabel('Score'); plt.ylim(0,1); plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout()
    plt.savefig(PLOT_DIR / f'{CURRENT_DECODER}_generation_metrics.png', dpi=180)
    plt.show()

select_metric = 'val_soft' if 'val_soft' in hist_df.columns else 'val_exact'
best_row = hist_df.sort_values(select_metric, ascending=False).iloc[0]
print('Best epoch selected by', select_metric)
display(pd.DataFrame([best_row]))
pd.DataFrame([best_row]).to_csv(MODEL_OUT_DIR / f'{CURRENT_DECODER}_best_epoch_summary.csv', index=False)

if set(['train_loss','val_loss']).issubset(hist_df.columns):
    gap = hist_df['val_loss'] - hist_df['train_loss']
    plt.figure(figsize=(9,4))
    plt.plot(hist_df['epoch'], gap, marker='o', color='crimson')
    plt.axhline(0, color='black', linewidth=1)
    plt.title(f'{CURRENT_MODEL_NAME} - Overfitting gap: val_loss - train_loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss gap'); plt.grid(True, alpha=0.3); plt.tight_layout()
    plt.savefig(PLOT_DIR / f'{CURRENT_DECODER}_overfitting_gap.png', dpi=180)
    plt.show()

heat_cols = [c for c in ['train_loss','val_loss','val_exact','val_soft','val_bleu','val_rougeL','val_meteor','val_llm_judge_rule'] if c in hist_df.columns]
plt.figure(figsize=(10, max(3, len(hist_df)*0.35)))
sns.heatmap(hist_df.set_index('epoch')[heat_cols], annot=True, fmt='.3f', cmap='YlGnBu')
plt.title(f'{CURRENT_MODEL_NAME} - Per-epoch metric heatmap')
plt.tight_layout(); plt.savefig(PLOT_DIR / f'{CURRENT_DECODER}_epoch_metric_heatmap.png', dpi=180); plt.show()

print('Saved plots to:', PLOT_DIR)
